# Chapter 3 (Interference) — figure generator

Runs every experiment behind the **Interference** chapter and saves the figures
with the exact filenames the chapter expects:

- `val_2d.pdf` — controlled 2D validation
- `within_batch_cifar10.pdf` — within-batch (Type A) interference on CIFAR-10
- `between_batch_cifar10.pdf` — between-batch (Type B) interference on CIFAR-10
- `scale_comparison.pdf` — metrics at scale (CIFAR-10 vs CIFAR-100)
- `deficit_correlations.pdf` — geometric indices vs the per-step deficit
- (optional) `within_batch_cosgd_effect.pdf`, `between_batch_bograd_effect.pdf`

**How to use.** Runtime → GPU (A100/L4 ideal; T4 fine). Run top to bottom. The
figures are written to Drive (if you mount it) and zipped for download at the end.
Then drop the PDFs into `dissertation/chapters/interference/figures/`.

It reuses the repo's real `interference` metrics module, so the figures match the
chapter's definitions exactly. Set `SMOKE = True` in the config cell for a ~5 min
end-to-end test first.

## 1. Clone the repo

In [ ]:
import os, subprocess

REPO_URL = "https://github.com/rayden96/MastersDissertationExperiments.git"
BRANCH   = "m0-infrastructure"   # change to your working branch
REPO_DIR = "/content/MastersDissertationExperiments"

if not os.path.exists(REPO_DIR):
    subprocess.run(["git", "clone", "--branch", BRANCH, REPO_URL, REPO_DIR], check=True)
else:
    subprocess.run(["git", "-C", REPO_DIR, "fetch", "origin"], check=True)
    subprocess.run(["git", "-C", REPO_DIR, "checkout", BRANCH], check=True)
    subprocess.run(["git", "-C", REPO_DIR, "pull", "origin", BRANCH], check=True)
print("repo at", REPO_DIR)
print(subprocess.run(["git", "-C", REPO_DIR, "log", "--oneline", "-1"],
                     capture_output=True, text=True).stdout)

## 2. Imports, device, config and helpers
`torch`/`torchvision`/`matplotlib` are preinstalled on Colab. We import the repo's
real interference module so the metrics are identical to the chapter's.

In [ ]:
import sys, time, json, math
import torch, numpy as np
import torch.nn as nn

sys.path.insert(0, REPO_DIR)
sys.path.insert(0, os.path.join(REPO_DIR, "PaperReadyExperiments"))

from interference import InterferenceMeter, summarize_run, correlate
from interference.image_runner import run_one_seed as run_image_seed, build_balanced_reference_subset

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if device.type == "cuda":
    torch.backends.cudnn.benchmark = True
    torch.set_float32_matmul_precision("high")
print("device:", device, "|", (torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"))
print("torch", torch.__version__)

In [ ]:
import matplotlib as mpl
import matplotlib.pyplot as plt

# ------------------------- CONFIG (edit me) -------------------------
SMOKE = False            # True = tiny everything (~5 min) to test the pipeline

SEEDS_2D   = [0, 1, 2, 3, 4]   # 2D controlled problem (instant)
N_STEPS_2D = 500

SEEDS_C10  = [2026, 2027, 2028]  # CIFAR-10 (within + between batch)
EPOCHS_C10 = 8
LR_C10     = 0.05

RUN_CIFAR100 = True              # CIFAR-100 ResNet-18 (the scale testbed; heavier)
SEEDS_C100   = [2026, 2027]
EPOCHS_C100  = 5
LR_C100      = 0.1

RUN_METHOD_EFFECTS = True        # show the metrics respond to COSGD / BOGrad
EPOCHS_EFFECT      = 8

K_VALUES    = [4, 32, 128]
BATCH       = 128
FIG_FORMATS = ["pdf"]            # add "png" for quick-preview copies
# -------------------------------------------------------------------

if SMOKE:
    SEEDS_2D = [0, 1]; N_STEPS_2D = 200
    SEEDS_C10 = [2026]; EPOCHS_C10 = 1
    RUN_CIFAR100 = True; SEEDS_C100 = [2026]; EPOCHS_C100 = 1
    RUN_METHOD_EFFECTS = True; EPOCHS_EFFECT = 1

try:
    from google.colab import drive
    drive.mount("/content/drive")
    FIG_DIR = "/content/drive/MyDrive/dissertation/interference_figures"
except Exception:
    FIG_DIR = "/content/interference_figures"
os.makedirs(FIG_DIR, exist_ok=True)
DATA_ROOT = os.path.join(REPO_DIR, "data")
print("figures ->", FIG_DIR)

mpl.rcParams.update({
    "figure.dpi": 120, "savefig.dpi": 300, "savefig.bbox": "tight",
    "font.family": "serif", "font.size": 11,
    "axes.titlesize": 11, "axes.labelsize": 11, "legend.fontsize": 9,
    "axes.grid": True, "grid.alpha": 0.3,
})

def save_fig(fig, name):
    for ext in FIG_FORMATS:
        p = os.path.join(FIG_DIR, name + "." + ext)
        fig.savefig(p)
        print("  wrote", p)

def series_mean_std(seed_logs, key):
    per_seed = []
    for logs in seed_logs:
        steps = [l["step"] for l in logs if key in l and l[key] == l[key]]
        vals  = [l[key]    for l in logs if key in l and l[key] == l[key]]
        per_seed.append((np.array(steps), np.array(vals, dtype=np.float64)))
    if not per_seed or any(s.size == 0 for s, _ in per_seed):
        return np.array([]), np.array([]), np.array([])
    common = set(per_seed[0][0].tolist())
    for s, _ in per_seed[1:]:
        common &= set(s.tolist())
    common = sorted(common)
    if not common:
        return np.array([]), np.array([]), np.array([])
    aligned = []
    for s, v in per_seed:
        idx = np.array([np.where(s == cs)[0][0] for cs in common])
        aligned.append(v[idx])
    M = np.stack(aligned, axis=0)
    return np.array(common), M.mean(axis=0), M.std(axis=0)

def load_module(name, relpath):
    import importlib.util
    path = os.path.join(REPO_DIR, relpath)
    spec = importlib.util.spec_from_file_location(name, path)
    mod = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(mod)
    return mod

def sgd_factory(lr, momentum=0.9):
    return lambda model: torch.optim.SGD(model.parameters(), lr=lr, momentum=momentum)

print("config ready | SMOKE =", SMOKE)

## 3. Part A — controlled 2D problem  ->  `val_2d.pdf`
The Gaussian-mixture toy makes interference visible. Far from the optimum the
per-class pulls agree; at the centroid (the optimum) they maximally conflict, so
`I_inter` falls and the mean cosine falls as the trajectory converges. The deficit
shrinks near the optimum because the reference-gradient magnitude vanishes there.

In [ ]:
m2d = load_module("mixture2d", "PaperReadyExperiments/01_small_2d/problem.py")
Mixture2DProblem = m2d.Mixture2DProblem

LR_2D = 0.2   # 2D zig-zag amplitude: higher lr => more oscillation near the centroid

def run_2d(seed, lr=LR_2D, n_steps=500, n_groups=10, R=2.0, batch_size=20,
           init_position=(3.0, 0.0)):
    prob = Mixture2DProblem(n_groups=n_groups, R=R, batch_size=batch_size, lr=lr,
                            init_position=init_position, seed=seed)
    meter = InterferenceMeter(problem=prob, lr=lr, K_values=K_VALUES,
                              log_every=1, ref_refresh_every=10)
    meter.initialize()
    traj = np.empty((n_steps + 1, 2), dtype=np.float32)
    traj[0] = prob.flatten_params().numpy()
    for step in range(n_steps):
        meter.before_step()
        batch, loss = prob.step_sgd()
        meter.after_step(step, batch, loss)
        traj[step + 1] = prob.flatten_params().numpy()
    return {"logs": meter.logs, "traj": traj,
            "centres": np.array(prob.config()["centres"], dtype=np.float32)}

runs2d = [run_2d(s, lr=LR_2D, n_steps=N_STEPS_2D) for s in SEEDS_2D]
logs2d = [r["logs"] for r in runs2d]
print("2D done:", len(runs2d), "seeds,", len(logs2d[0]), "logged steps each | lr =", LR_2D)

def loss_grid(centres, xs, ys):
    pts = np.stack(np.meshgrid(xs, ys, indexing="xy"), axis=-1)
    diffs = pts[:, :, None, :] - centres[None, None, :, :]
    return 0.5 * (diffs ** 2).sum(axis=-1).mean(axis=-1)

fig, axs = plt.subplots(2, 2, figsize=(11, 9))
centres = runs2d[0]["centres"]
allp = np.concatenate([r["traj"] for r in runs2d], axis=0)
xmin, ymin = allp.min(0) - 0.4; xmax, ymax = allp.max(0) + 0.4
xmin = min(xmin, centres[:, 0].min() - 0.4, -0.4); xmax = max(xmax, centres[:, 0].max() + 0.4, 0.4)
ymin = min(ymin, centres[:, 1].min() - 0.4, -0.4); ymax = max(ymax, centres[:, 1].max() + 0.4, 0.4)
xs = np.linspace(xmin, xmax, 140); ys = np.linspace(ymin, ymax, 140)
Z = loss_grid(centres, xs, ys)
ax = axs[0, 0]
ax.contour(xs, ys, Z, levels=12, colors="0.6", linewidths=0.6, alpha=0.8)
ax.contourf(xs, ys, Z, levels=20, cmap="viridis", alpha=0.25)
ax.scatter(centres[:, 0], centres[:, 1], marker="x", s=55, c="black", label="class centres")
ax.scatter([0], [0], marker="*", s=140, c="red", label="optimum")
for r in runs2d:
    t = r["traj"]; ax.plot(t[:, 0], t[:, 1], lw=1.0, alpha=0.8)
ax.scatter([runs2d[0]["traj"][0, 0]], [runs2d[0]["traj"][0, 1]], marker="o", s=35,
           c="white", edgecolors="black", zorder=5, label="start")
ax.set_aspect("equal"); ax.set_xlabel(r"$\theta_1$"); ax.set_ylabel(r"$\theta_2$")
ax.set_title("(a) Trajectory on the 2D loss surface"); ax.legend(loc="upper right", fontsize=8); ax.grid(False)

# (b) I_inter and (d) D_t via the simple loop
for ax, key, title, yl in [
    (axs[0, 1], "I_inter", r"(b) $I_{\mathrm{inter}}(t)$",        r"$I_{\mathrm{inter}}$"),
    (axs[1, 1], "D_t",     r"(d) Per-step deficit $D_t$",          r"$D_t$"),
]:
    st, mu, sd = series_mean_std(logs2d, key)
    ax.plot(st, mu, color="tab:blue", lw=1.5)
    ax.fill_between(st, mu - sd, mu + sd, color="tab:blue", alpha=0.2)
    ax.axhline(0.0, color="0.5", lw=0.6)
    ax.set_xlabel("step"); ax.set_ylabel(yl); ax.set_title(title)

# (c) within-batch mean cosine: aligning vs all vs conflicting on one axis
axc = axs[1, 0]
for key, col, lab in [("inter_mean_cos_pos", "tab:blue",  "aligning (cos>0)"),
                      ("inter_mean_cos",     "tab:green", "all (signed)"),
                      ("inter_mean_cos_neg", "tab:red",   "conflicting (cos<0)")]:
    st, mc, sc = series_mean_std(logs2d, key)
    if st.size:
        axc.plot(st, mc, color=col, lw=1.4, label=lab)
        axc.fill_between(st, mc - sc, mc + sc, color=col, alpha=0.12)
axc.axhline(0.0, color="0.5", lw=0.6)
axc.set_xlabel("step"); axc.set_ylabel("mean cosine")
axc.set_title("(c) Within-batch mean pairwise cosine"); axc.legend(fontsize=7, loc="best")

fig.suptitle(r"Validation on a controlled 2D Gaussian-mixture problem (mean $\pm$ std across seeds)", y=0.99)
fig.tight_layout(rect=(0, 0, 1, 0.97))
save_fig(fig, "val_2d"); plt.show()

## 4. Part B — CIFAR-10 within- and between-batch
Baseline SGD+momentum with the interference meter attached, multiple seeds. This is
the empirical evidence that both types of interference are present and measurable in
a real network.

In [ ]:
m02 = load_module("c10mod", "PaperReadyExperiments/02_medium_cifar10/run.py")
SmallCIFARCNN = m02.SmallCIFARCNN
build_cifar10 = m02.build_cifar10

train_c10, test_c10 = build_cifar10(DATA_ROOT)
ref_c10 = build_balanced_reference_subset(train_c10, num_classes=10, n_per_class=200, seed=2026)

c10_logs, c10_summaries = [], []
for s in SEEDS_C10:
    res = run_image_seed(
        seed=s, model_factory=lambda: SmallCIFARCNN(num_classes=10),
        train_dataset=train_c10, test_dataset=test_c10, ref_dataset=ref_c10,
        num_classes=10, device=device, optimizer_factory=sgd_factory(LR_C10),
        lr=LR_C10, epochs=EPOCHS_C10, batch_size=BATCH,
        log_every=50, ref_refresh_every=100, K_values=K_VALUES, num_workers=2,
    )
    c10_logs.append(res["logs"]); c10_summaries.append(res["summary"])
print("CIFAR-10 baseline done:", len(c10_logs), "seeds")

In [ ]:
# within_batch_cifar10.pdf
fig, axs = plt.subplots(1, 3, figsize=(15, 4.2))
st, mu, sd = series_mean_std(c10_logs, "I_inter")
axs[0].plot(st, mu, color="tab:blue", lw=1.5); axs[0].fill_between(st, mu - sd, mu + sd, alpha=0.2, color="tab:blue")
axs[0].axhline(1.0, color="0.5", lw=0.6, ls="--")
axs[0].set_title(r"(a) $I_{\mathrm{inter}}(t)$"); axs[0].set_xlabel("step"); axs[0].set_ylabel(r"$I_{\mathrm{inter}}$")

# (b) angle structure: aligning vs conflicting vs signed mean, plus conflicting share
for key, col, lab in [("inter_mean_cos_pos", "tab:blue",  "aligning (cos>0)"),
                      ("inter_mean_cos",     "tab:green", "all pairs (signed)"),
                      ("inter_mean_cos_neg", "tab:red",   "conflicting (cos<0)")]:
    st, mc, sc = series_mean_std(c10_logs, key)
    if st.size:
        axs[1].plot(st, mc, color=col, lw=1.4, label=lab)
        axs[1].fill_between(st, mc - sc, mc + sc, alpha=0.12, color=col)
axs[1].axhline(0.0, color="0.5", lw=0.6); axs[1].set_xlabel("step"); axs[1].set_ylabel("mean pairwise cosine")
ax1b = axs[1].twinx()
st2, mf, sf = series_mean_std(c10_logs, "inter_frac_neg")
ax1b.plot(st2, mf, color="0.35", lw=1.1, ls="--", label="frac. conflicting")
ax1b.set_ylabel("frac. pairs with cos < 0"); ax1b.set_ylim(0, 1); ax1b.grid(False)
axs[1].set_title("(b) Within-batch angle structure")
l1, la1 = axs[1].get_legend_handles_labels(); l2, la2 = ax1b.get_legend_handles_labels()
axs[1].legend(l1 + l2, la1 + la2, fontsize=7, loc="best")

st, mu, sd = series_mean_std(c10_logs, "inter_useful_descent_frac")
axs[2].plot(st, mu, color="tab:purple", lw=1.5); axs[2].fill_between(st, mu - sd, mu + sd, alpha=0.2, color="tab:purple")
axs[2].axhline(1.0, color="0.5", lw=0.6, ls="--")
axs[2].set_title(r"(c) Useful descent fraction of $g_t$"); axs[2].set_xlabel("step"); axs[2].set_ylabel("useful descent fraction")
fig.suptitle(r"Within-batch interference on CIFAR-10 (mean $\pm$ std across seeds)", y=1.02)
fig.tight_layout(); save_fig(fig, "within_batch_cifar10"); plt.show()

In [ ]:
# between_batch_cifar10.pdf
colors = {4: "tab:blue", 32: "tab:orange", 128: "tab:green"}
fig, axs = plt.subplots(1, 3, figsize=(15, 4.2))
for K in K_VALUES:
    st, mu, sd = series_mean_std(c10_logs, "I_between_K%d" % K)
    if st.size:
        axs[0].plot(st, mu, lw=1.5, color=colors.get(K), label="K=%d" % K)
        axs[0].fill_between(st, mu - sd, mu + sd, alpha=0.15, color=colors.get(K))
axs[0].axhline(1.0, color="0.5", lw=0.6, ls="--")
axs[0].set_title(r"(a) $I_{\mathrm{between},K}(t)$"); axs[0].set_xlabel("step"); axs[0].set_ylabel(r"$I_{\mathrm{between},K}$"); axs[0].legend(fontsize=8)
for K in K_VALUES:
    st, mu, sd = series_mean_std(c10_logs, "between_K%d_useful_path_frac" % K)
    if st.size:
        axs[1].plot(st, mu, lw=1.5, color=colors.get(K), label="K=%d" % K)
        axs[1].fill_between(st, mu - sd, mu + sd, alpha=0.15, color=colors.get(K))
axs[1].set_title("(b) Useful path fraction"); axs[1].set_xlabel("step"); axs[1].set_ylabel("useful path fraction"); axs[1].legend(fontsize=8)

# (c) within-window mean cosine at the headline K: aligning vs all vs conflicting
KC = 32
for key, col, lab in [("between_K%d_mean_cos_pos" % KC, "tab:blue",  "aligning (cos>0)"),
                      ("between_K%d_mean_cos" % KC,     "tab:green", "all (signed)"),
                      ("between_K%d_mean_cos_neg" % KC, "tab:red",   "conflicting (cos<0)")]:
    st, mu, sd = series_mean_std(c10_logs, key)
    if st.size:
        axs[2].plot(st, mu, lw=1.4, color=col, label=lab)
        axs[2].fill_between(st, mu - sd, mu + sd, alpha=0.12, color=col)
axs[2].axhline(0.0, color="0.5", lw=0.6)
axs[2].set_title("(c) Within-window mean cosine (K=%d)" % KC); axs[2].set_xlabel("step"); axs[2].set_ylabel("mean pairwise cosine"); axs[2].legend(fontsize=8)
fig.suptitle(r"Between-batch interference on CIFAR-10 (mean $\pm$ std across seeds)", y=1.02)
fig.tight_layout(); save_fig(fig, "between_batch_cifar10"); plt.show()

In [ ]:
# between_lagcos_cifar10.pdf : lag-resolved update autocorrelation cos(u_t, u_{t-k})
# Compares plain SGD (momentum 0) with SGD+momentum. Negative at small lag =
# cancellation (steps oppose); positive & slowly decaying = curvature (steps
# cooperate, direction rotates). Momentum smooths the short-lag cancellation, so the
# comparison both shows the between-batch interference exists and what already damps it.
def lag_profile(seed_logs):
    rows = [np.asarray(l["between_lagcos"], dtype=np.float64)
            for logs in seed_logs for l in logs if l.get("between_lagcos") is not None]
    if not rows:
        return np.array([]), np.array([]), np.array([])
    M = np.vstack(rows); lags = np.arange(1, M.shape[1] + 1)
    mu, sd = np.nanmean(M, axis=0), np.nanstd(M, axis=0)
    keep = ~np.isnan(mu)
    return lags[keep], mu[keep], sd[keep]

# plain-SGD (momentum 0) companion run, so the short-lag cancellation is visible
nomom_logs = []
for s in SEEDS_C10:
    res = run_image_seed(
        seed=s, model_factory=lambda: SmallCIFARCNN(num_classes=10),
        train_dataset=train_c10, test_dataset=test_c10, ref_dataset=ref_c10,
        num_classes=10, device=device,
        optimizer_factory=lambda model: torch.optim.SGD(model.parameters(), lr=LR_C10, momentum=0.0),
        lr=LR_C10, epochs=EPOCHS_C10, batch_size=BATCH,
        log_every=50, ref_refresh_every=100, K_values=K_VALUES, num_workers=2,
    )
    nomom_logs.append(res["logs"])
print("plain-SGD companion done:", len(nomom_logs), "seeds")

fig, ax = plt.subplots(figsize=(7.8, 4.6))
for name, slogs, col in [("plain SGD (no momentum)", nomom_logs, "tab:red"),
                         ("SGD + momentum 0.9", c10_logs, "tab:blue")]:
    lags, mu, sd = lag_profile(slogs)
    if lags.size:
        ax.plot(lags, mu, color=col, lw=1.7, label=name)
        ax.fill_between(lags, mu - sd, mu + sd, color=col, alpha=0.13)
ax.axhline(0.0, color="0.5", lw=0.8)
for Kref in K_VALUES:
    ax.axvline(Kref, color="0.78", lw=0.7, ls=":")
ax.set_xscale("log")
ax.set_xlabel(r"lag $k$ (steps)"); ax.set_ylabel(r"mean $\cos(u_t,\, u_{t-k})$")
ax.set_title("Lag-resolved update autocorrelation on CIFAR-10\n"
             "(negative at small lag = cancellation; positive & decaying = curvature; "
             "dotted = $K\\in\\{4,32,128\\}$)")
ax.legend(fontsize=9)
fig.tight_layout(); save_fig(fig, "between_lagcos_cifar10"); plt.show()

## 5. Part C — CIFAR-100 ResNet-18 (behaviour at scale)
Heavier (BatchNorm, 100 classes -> 100 masked backward passes per logged step). Set
`RUN_CIFAR100 = False` to skip. Produces `scale_comparison.pdf`.

In [ ]:
c100_logs, c100_summaries = [], []
if RUN_CIFAR100:
    m03 = load_module("c100mod", "PaperReadyExperiments/03_large_cifar100/run.py")
    ResNet18CIFAR = m03.ResNet18CIFAR
    build_cifar100 = m03.build_cifar100
    train_c100, test_c100 = build_cifar100(DATA_ROOT)
    ref_c100 = build_balanced_reference_subset(train_c100, num_classes=100, n_per_class=100, seed=2026)
    for s in SEEDS_C100:
        res = run_image_seed(
            seed=s, model_factory=lambda: ResNet18CIFAR(num_classes=100),
            train_dataset=train_c100, test_dataset=test_c100, ref_dataset=ref_c100,
            num_classes=100, device=device, optimizer_factory=sgd_factory(LR_C100),
            lr=LR_C100, epochs=EPOCHS_C100, batch_size=BATCH,
            log_every=100, ref_refresh_every=200, K_values=K_VALUES, num_workers=2,
        )
        c100_logs.append(res["logs"]); c100_summaries.append(res["summary"])
    print("CIFAR-100 baseline done:", len(c100_logs), "seeds")
else:
    print("RUN_CIFAR100 = False; skipping scale testbed")

In [ ]:
# scale_comparison.pdf : rows = dataset, cols = metric
panels = [("I_inter", r"$I_{\mathrm{inter}}$"), ("I_between_K32", r"$I_{\mathrm{between},32}$"),
          ("inter_useful_descent_frac", "useful descent frac"), ("D_t", r"$D_t$")]
rows = [("CIFAR-10 CNN", c10_logs, "tab:blue")]
if RUN_CIFAR100 and c100_logs:
    rows.append(("CIFAR-100 ResNet-18", c100_logs, "tab:red"))
fig, axs = plt.subplots(len(rows), 4, figsize=(16, 3.6 * len(rows)), squeeze=False)
for ri, (rname, rlogs, rc) in enumerate(rows):
    for ci, (key, yl) in enumerate(panels):
        ax = axs[ri][ci]
        st, mu, sd = series_mean_std(rlogs, key)
        if st.size:
            ax.plot(st, mu, color=rc, lw=1.4); ax.fill_between(st, mu - sd, mu + sd, alpha=0.2, color=rc)
        if key == "D_t":
            ax.axhline(0.0, color="0.5", lw=0.6)
        ax.set_xlabel("step")
        ax.set_ylabel(rname if ci == 0 else yl)
        if ri == 0:
            ax.set_title(yl)
fig.suptitle(r"Interference metrics at scale (mean $\pm$ std across seeds)", y=1.0)
fig.tight_layout(); save_fig(fig, "scale_comparison"); plt.show()

## 6. Part D — do the geometric indices predict the deficit?  ->  `deficit_correlations.pdf`
Within-run Pearson correlation between each geometric index and the per-step deficit
`D_t`, pooled across seeds, one column per testbed. Expected sign: **negative** (less
cancellation -> less training-hurt). Where it is not negative, the framework is
flagging a magnitude/curvature confound (Section 3.5.3) rather than failing.

In [ ]:
metrics = [
    ("I_inter", r"$I_{\mathrm{inter}}$"),
    ("inter_mean_cos", "within-batch mean cos"),
    ("inter_useful_descent_frac", "useful descent frac"),
    ("I_between_K4", r"$I_{\mathrm{between},4}$"),
    ("I_between_K32", r"$I_{\mathrm{between},32}$"),
    ("I_between_K128", r"$I_{\mathrm{between},128}$"),
    ("between_K32_useful_path_frac", "useful path frac (K=32)"),
]
testbeds = [("2D", logs2d), ("CIFAR-10", c10_logs)]
if RUN_CIFAR100 and c100_logs:
    testbeds.append(("CIFAR-100", c100_logs))

def pooled(seed_logs):
    out = []
    for logs in seed_logs:
        out.extend(logs)
    return out

M = np.full((len(metrics), len(testbeds)), np.nan)
for j, (tname, slogs) in enumerate(testbeds):
    pl = pooled(slogs)
    for i, (key, _) in enumerate(metrics):
        M[i, j] = correlate(pl, key, "D_t")

fig, ax = plt.subplots(figsize=(1.7 * len(testbeds) + 4.0, 0.62 * len(metrics) + 1.6))
im = ax.imshow(M, cmap="RdBu_r", vmin=-1, vmax=1, aspect="auto")
ax.set_xticks(range(len(testbeds))); ax.set_xticklabels([t[0] for t in testbeds])
ax.set_yticks(range(len(metrics))); ax.set_yticklabels([m[1] for m in metrics])
for i in range(len(metrics)):
    for j in range(len(testbeds)):
        v = M[i, j]
        if v == v:
            ax.text(j, i, "%.2f" % v, ha="center", va="center", fontsize=9,
                    color=("white" if abs(v) > 0.6 else "black"))
ax.set_title("Pearson r between geometric index and per-step deficit $D_t$\n(negative = less cancellation accompanies less training-hurt)", fontsize=10)
ax.grid(False)
fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04, label="Pearson r")
fig.tight_layout(); save_fig(fig, "deficit_correlations"); plt.show()

import csv
with open(os.path.join(FIG_DIR, "val_correlations.csv"), "w", newline="") as f:
    w = csv.writer(f); w.writerow(["metric"] + [t[0] for t in testbeds])
    for i, (key, lab) in enumerate(metrics):
        w.writerow([key] + ["%.4f" % M[i, j] for j in range(len(testbeds))])
print("  wrote", os.path.join(FIG_DIR, "val_correlations.csv"))

## 7. Part E (optional) — the metrics respond to the methods
A quick 1-seed check that COSGD raises the within-batch metrics and BOGrad raises
the between-batch metrics, plus the deficit-bias illustration (BOGrad shortens steps,
so its cumulative deficit looks worse even when it helps). Produces
`within_batch_cosgd_effect.pdf` and `between_batch_bograd_effect.pdf`. Set
`RUN_METHOD_EFFECTS = False` to skip.

In [ ]:
if RUN_METHOD_EFFECTS:
    from common.optimizers import BoGrad, COSGD
    crit = nn.CrossEntropyLoss()

    cosgd_factory = lambda model: COSGD(model.parameters(), lr=LR_C10, model=model, criterion=crit,
                                        orthogonalization_method="modified_gs_negative", step_method="single_forward")
    common_kw = dict(train_dataset=train_c10, test_dataset=test_c10, ref_dataset=ref_c10, num_classes=10,
                     device=device, lr=LR_C10, epochs=EPOCHS_EFFECT, batch_size=BATCH,
                     log_every=50, ref_refresh_every=100, K_values=K_VALUES, num_workers=2)
    base1 = run_image_seed(seed=2026, model_factory=lambda: SmallCIFARCNN(10),
                           optimizer_factory=sgd_factory(LR_C10), **common_kw)
    cos1  = run_image_seed(seed=2026, model_factory=lambda: SmallCIFARCNN(10),
                           optimizer_factory=cosgd_factory, is_cosgd=True, **common_kw)

    fig, axs = plt.subplots(1, 2, figsize=(10.5, 4.2))
    for lab, r, c in [("baseline SGD", base1, "tab:gray"), ("COSGD", cos1, "tab:blue")]:
        for ax, key in [(axs[0], "I_inter"), (axs[1], "inter_useful_descent_frac")]:
            st, mu, sd = series_mean_std([r["logs"]], key); ax.plot(st, mu, lw=1.6, color=c, label=lab)
    axs[0].set_title(r"$I_{\mathrm{inter}}(t)$"); axs[0].set_xlabel("step"); axs[0].legend(fontsize=8)
    axs[1].set_title("useful descent fraction"); axs[1].set_xlabel("step"); axs[1].legend(fontsize=8)
    fig.suptitle("COSGD raises the within-batch metrics (1 seed, CIFAR-10)", y=1.02)
    fig.tight_layout(); save_fig(fig, "within_batch_cosgd_effect"); plt.show()

    bograd_factory = lambda model: BoGrad(model.parameters(), base_optimizer_cls=torch.optim.SGD,
                                          buffer_size=32, project_stage="update", projection_mode="negative",
                                          orth_method="sequential", collect_stats=False, lr=LR_C10, momentum=0.9)
    bog1 = run_image_seed(seed=2026, model_factory=lambda: SmallCIFARCNN(10),
                          optimizer_factory=bograd_factory, **common_kw)

    fig, axs = plt.subplots(1, 3, figsize=(15, 4.2))
    for lab, r, c in [("baseline SGD", base1, "tab:gray"), ("BOGrad", bog1, "tab:orange")]:
        st, mu, sd = series_mean_std([r["logs"]], "I_between_K32"); axs[0].plot(st, mu, lw=1.6, color=c, label=lab)
        st, mu, sd = series_mean_std([r["logs"]], "between_K32_useful_path_frac"); axs[1].plot(st, mu, lw=1.6, color=c, label=lab)
    axs[0].set_title(r"$I_{\mathrm{between},32}(t)$"); axs[0].set_xlabel("step"); axs[0].legend(fontsize=8)
    axs[1].set_title("useful path fraction (K=32)"); axs[1].set_xlabel("step"); axs[1].legend(fontsize=8)
    names = ["baseline", "BOGrad"]
    cum = [base1["summary"].get("cum_deficit", float("nan")), bog1["summary"].get("cum_deficit", float("nan"))]
    acc = [base1["summary"].get("best_test_acc", float("nan")), bog1["summary"].get("best_test_acc", float("nan"))]
    axb = axs[2]; x = np.arange(2)
    axb.bar(x - 0.2, cum, width=0.4, color="tab:purple", label="cum. deficit"); axb.grid(False)
    axb.set_xticks(x); axb.set_xticklabels(names); axb.set_ylabel("cumulative deficit")
    axt = axb.twinx(); axt.plot(x, acc, "o-", color="tab:green", label="best test acc"); axt.set_ylabel("best test acc"); axt.grid(False)
    axb.set_title("(c) Deficit bias: BOGrad shortens steps")
    fig.suptitle("BOGrad raises between-batch metrics; cumulative deficit is biased against it (1 seed)", y=1.02)
    fig.tight_layout(); save_fig(fig, "between_batch_bograd_effect"); plt.show()
else:
    print("RUN_METHOD_EFFECTS = False; skipping method-effect figures")

## 8. Package the figures
Zips everything in `FIG_DIR` and downloads it. Move the PDFs into
`dissertation/chapters/interference/figures/` (the chapter includes them by exact
filename, with an `\IfFileExists` fallback so it builds either way).

In [ ]:
import shutil
zip_base = "/content/interference_figures"
shutil.make_archive(zip_base, "zip", FIG_DIR)
print("zipped ->", zip_base + ".zip")
print("\nFiles in", FIG_DIR, ":")
for f in sorted(os.listdir(FIG_DIR)):
    print("  ", f)
print("\nNext: move the .pdf files into dissertation/chapters/interference/figures/")
try:
    from google.colab import files
    files.download(zip_base + ".zip")
except Exception as e:
    print("(download manually from the Files pane)", e)